# Part 5 — Continue normal pipeline only

**Degraded gate ผ่านการพิสูจน์แล้ว ไม่ต้อง train ชุดนั้นซ้ำ**
ค่าที่วัดจริง: validation PR-AUC 0.1008333, ต่ำกว่า floor 0.4
ingest/validate/train completed, evaluate failed, registry ไม่เพิ่ม และไม่มี endpoint
ตอนตรวจ 15:01 น. compute ทั้งสองชุดเหลือ 0 nodes

cell นี้จะ:
1. ตรวจผล inspection และ hash ของ source bundle เดิม
2. อ่านตรวจ identities ที่ผูกกับ compute เดิม โดยไม่สร้าง/แก้ resource หรือ role
3. บันทึก gate-failure evidence จาก job เดิม
4. ส่ง **normal pipeline** ชื่อ `lab5-20261023-070538-normal` ถ้ายังไม่มี หรืออ่านติดตามงานชื่อนี้ถ้ามีอยู่แล้ว
5. เมื่อ gate ผ่าน จึง register/deploy staging และ smoke test ตาม DAG เดิม

**มีค่า compute/training/staging ตามแผนเดิม แต่ไม่มี setup, degraded retraining, Git commit หรือ push**
ใช้ WSL/kernel `itcs355-lab3` เดิม รันเฉพาะ cell ด้านล่าง ไม่ต้องรัน main Part 5

ตัวอ่านสถานะแก้แล้ว: Failed/Completed จะหยุดรอทันที
รอ Azure provisioning ได้จาก progress/job status ทุก 15s; ไม่กด cell ซ้อนกัน
หาก error ส่ง output พร้อม job ID กลับมา ไม่สร้าง job ชื่อใหม่เอง
Interrupt หยุดรอในเครื่อง ไม่ได้ cancel งานบน Azure

หาก normal ผ่าน จะมี staging endpoint เปิดอยู่ ต้องเก็บหลักฐาน/cleanup ต่อ


In [1]:
from pathlib import Path
import platform
import subprocess
import sys

if platform.system() != "Linux" or "microsoft" not in platform.release().lower():
    raise RuntimeError("เลือก kernel itcs355-lab3 ฝั่ง WSL ก่อนครับ")
here = Path.cwd().resolve()
lab = next((base / "lab5" for base in (here, *here.parents)
            if (base / "lab5/scripts/resume_normal.py").is_file()
            and (base / "lab4/README.md").is_file()), None)
if lab is None:
    raise RuntimeError("เปิด notebook จาก repository เดิมก่อนครับ")
print("Continue normal pipeline only — reuse existing resources; no degraded retraining.", flush=True)
with subprocess.Popen(
    [sys.executable, "-u", "-m", "scripts.resume_normal"], cwd=lab,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
) as task:
    try:
        for line in task.stdout:
            print(line, end="", flush=True)
        exit_code = task.wait()
    except KeyboardInterrupt:
        task.terminate()
        try:
            task.wait(timeout=10)
        except subprocess.TimeoutExpired:
            task.kill()
        print("Azure job may continue. Inspect Studio; do not start a duplicate.", flush=True)
        raise
if exit_code:
    raise RuntimeError("Normal pipeline หยุด: ส่ง summary/error พร้อม job ID ครับ ไม่ต้องกลับไปเริ่มใหม่")
print("ส่ง SEND THIS SUMMARY กลับมาครับ staging ยังเปิดอยู่และต้อง cleanup ต่อ", flush=True)


Continue normal pipeline only — reuse existing resources; no degraded retraining.
[15:09:35] 1/3 Verify inspected gate failure and the unchanged runtime bundle
[15:09:52] 2/3 Read existing job/resources; skip identity, role and compute setup
[15:10:02] 3/3 Run/observe normal pipeline only: lab5-20261023-070538-normal
Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an 

RuntimeError: Normal pipeline หยุด: ส่ง summary/error พร้อม job ID ครับ ไม่ต้องกลับไปเริ่มใหม่